In [ ]:
# Cell 1 — mount, clone, symlink, deps
from google.colab import drive
drive.mount('/content/drive')

import pathlib, sys
DRIVE = pathlib.Path('/content/drive/MyDrive/pcb-defect')
%cd /content
!rm -rf /content/pcb-defect
!git clone https://github.com/akshatyuvan/pcb-defect.git
%cd /content/pcb-defect
!rm -rf data artifacts
!ln -s {DRIVE}/data data
!ln -s {DRIVE}/artifacts artifacts
!pip -q install "mlflow>=2.16"
sys.path.insert(0, '/content/pcb-defect')

# Day 1's outputs must already be here. If these listings are empty, Day 1 did
# not persist to Drive and must be rerun.
!ls data/patches
!ls artifacts

In [ ]:
# Cell 2 — the MLflow SQLite db lives on LOCAL disk, not Drive.
# SQLite over the Drive FUSE mount has unreliable file locking and will
# intermittently throw "database is locked" partway through a run. We write
# locally and snapshot to Drive after each run instead.
import shutil, pathlib
DB_LOCAL = '/content/mlflow.db'
DB_DRIVE = '/content/drive/MyDrive/pcb-defect/mlflow.db'

if pathlib.Path(DB_DRIVE).exists():
    shutil.copy(DB_DRIVE, DB_LOCAL)      # resume a previous session's runs
    print('restored existing mlflow.db from Drive')
else:
    print('starting a fresh mlflow.db')

def snapshot():
    shutil.copy(DB_LOCAL, DB_DRIVE)
    print('mlflow db snapshotted to Drive')

In [ ]:
# Cell 3 — RUN 1: unweighted CE. The ablation control.
!python -m src.train --run-name r1_unweighted --no-weighted --epochs 30
snapshot()

In [ ]:
# Cell 4 — RUN 2: inverse-frequency weighted CE
!python -m src.train --run-name r2_weighted --epochs 30
snapshot()

In [ ]:
# Cell 5 — RUN 3: weighted CE + label-preserving geometric augmentation
!python -m src.train --run-name r3_weighted_aug --augment --epochs 30
snapshot()

In [ ]:
# Cell 5b — RUN 4: partial weighting. Run 1 (no weighting) beat Run 2 (full
# inverse frequency) on both macro F1 and precision at the operating point, so
# the interesting question is whether the useful amount of weighting is zero or
# merely less than 1.0. sqrt weighting gives ~7x on good instead of ~50x.
!python -m src.train --run-name r4_weighted_p05 --weight-power 0.5 --epochs 30
snapshot()

In [ ]:
# Cell 6 — comparison table first
import mlflow, pandas as pd
mlflow.set_tracking_uri('sqlite:////content/mlflow.db')
df = mlflow.search_runs(experiment_names=['pcb-patch-classification'])
cols = ['tags.mlflow.runName','params.weight_power','params.augment',
        'metrics.best_val_macro_f1','metrics.test_macro_f1',
        'metrics.test_accuracy','metrics.op_precision','metrics.op_threshold']
display(df[cols].sort_values('metrics.best_val_macro_f1', ascending=False))

In [ ]:
# Cell 7 — register the macro-F1 winner. Rerunning rather than registering
# retrospectively keeps the registered artifact and the logged params provably
# from the same execution.
!python -m src.train --run-name r4_weighted_p05_registered --weight-power 0.5 --epochs 30 --register
snapshot()

In [ ]:
# Cell 8 — the registry is a hard Day 5 dependency. Confirm it NOW, not then.
from mlflow.tracking import MlflowClient
c = MlflowClient(tracking_uri='sqlite:////content/mlflow.db')
m = c.get_registered_model('pcbnet')
print('registered model:', m.name)
for v in c.search_model_versions("name='pcbnet'"):
    print('  version', v.version, '| run', v.run_id)

In [ ]:
# Cell 8b — dedup. Re-running the notebook end-to-end logged each config three
# times, plus two crashed runs. Keep the NEWEST FINISHED run per name so every
# survivor comes from the current code, including the input_example fix.
from mlflow.tracking import MlflowClient

c = MlflowClient(tracking_uri='sqlite:////content/mlflow.db')
exp = c.get_experiment_by_name('pcb-patch-classification')
runs = c.search_runs([exp.experiment_id], run_view_type=1,   # 1 = ACTIVE_ONLY
                     order_by=["attributes.start_time DESC"])

seen, deleted = set(), 0
for r in runs:
    name = r.data.tags.get('mlflow.runName')
    if r.info.status != 'FINISHED' or name in seen:
        c.delete_run(r.info.run_id)
        deleted += 1
        print(f'deleted {r.info.status:8} {name}')
    else:
        seen.add(name)
        print(f'KEPT    {r.info.status:8} {name}')

print(f'\ndeleted {deleted}, kept {len(seen)}')

In [ ]:
# Cell 8c (revised) — include params.weighted. weight_power is logged
# unconditionally but is only READ when weighted=True, so on the r1 control it
# is a defaulted value that describes nothing. Without the weighted column the
# table implies three runs shared a configuration they did not share.
import mlflow
mlflow.set_tracking_uri('sqlite:////content/mlflow.db')
df = mlflow.search_runs(experiment_names=['pcb-patch-classification'])

# Effective weight power: 0.0 means no weighting at all, which is the honest
# third point on the ablation curve alongside 0.5 and 1.0.
df['effective_power'] = df.apply(
    lambda r: 0.0 if r['params.weighted'] == 'False' else float(r['params.weight_power']),
    axis=1)

print(df[['tags.mlflow.runName','params.weighted','effective_power','params.augment',
          'metrics.best_val_macro_f1','metrics.test_macro_f1',
          'metrics.op_precision','metrics.op_threshold']]
      .sort_values('metrics.best_val_macro_f1', ascending=False).to_string())

In [ ]:
# Cell 8d — deleting runs soft-deletes them, so a registered version could now
# point at a deleted run. Day 5 loads by version, so a dangling pointer there
# breaks the service.
from mlflow.tracking import MlflowClient
c = MlflowClient(tracking_uri='sqlite:////content/mlflow.db')
kept = {r.info.run_id for r in c.search_runs(
    [c.get_experiment_by_name('pcb-patch-classification').experiment_id],
    run_view_type=1)}
for v in c.search_model_versions("name='pcbnet'"):
    print(f'version {v.version} | run {v.run_id} | run still active: {v.run_id in kept}')

snapshot()

In [ ]:
# Cell 9 — zip artifacts for download. This is how results reach the repo, since
# we deliberately never push from Colab.
!cd /content/pcb-defect && zip -r /content/day2_artifacts.zip \
    artifacts/figures artifacts/*_report.txt artifacts/classes.json artifacts/norm.json

# The three files Day 5 needs on the Mac. Artifact paths recorded in this SQLite
# db are absolute /content/drive/... paths that will not resolve on macOS, so
# Day 5 re-registers into a local Mac registry from these three files. That is
# planned Day 5 work; today's job is simply confirming all three exist.
print('\n--- Day 5 carry-over files ---')
!ls -la artifacts/best.pt artifacts/classes.json artifacts/norm.json

from google.colab import files
files.download('/content/day2_artifacts.zip')

In [ ]:
!ls -la /content/drive/MyDrive/pcb-defect/mlflow.db
!ls -la /content/drive/MyDrive/pcb-defect/artifacts/

In [ ]:
# Confirm the runtime is ready and the db came back before doing anything else.
import pathlib
print('db present:', pathlib.Path('/content/mlflow.db').exists())
print('patches present:', pathlib.Path('data/patches/test.npz').exists())
print('checkpoint present:',
      pathlib.Path('artifacts/r4_weighted_p05_registered_best.pt').exists())
print('snapshot defined:', 'snapshot' in dir())

In [ ]:
# CPU is fine — this is a 1.17M-param forward pass on a single zero tensor.
import mlflow, torch
mlflow.set_tracking_uri('sqlite:////content/mlflow.db')
try:
    m = mlflow.pytorch.load_model('models:/pcbnet/1')
    out = m(torch.zeros(1, 1, 64, 64))
    print('LOAD OK — output shape', tuple(out.shape))   # expect (1, 7)
except Exception as e:
    print('LOAD FAILED:', type(e).__name__)
    print(str(e)[:600])

In [ ]:
# Eyeball 3 — what does the model get wrong on its weakest class?
# mousebite has the lowest recall in r4 (0.533): nearly half of all real
# mousebites are called something else. If several of these look mislabelled to
# a human, the suspect is --min-frac 0.25 rather than the model, and that
# becomes a Day 4 investigation.
import numpy as np, torch, json, matplotlib.pyplot as plt
import torch.nn.functional as F
from src.models.cnn import PCBNet
from src.data.deeppcb import CLASSES

d = np.load('data/patches/test.npz')
X, y = d['X'], d['y']
norm = json.load(open('artifacts/norm.json'))
ck = torch.load('artifacts/r4_weighted_p05_registered_best.pt', map_location='cpu')
model = PCBNet(num_classes=7); model.load_state_dict(ck['state_dict']); model.eval()

TARGET = CLASSES.index('mousebite')
idx = np.flatnonzero(y == TARGET)
Xt = torch.from_numpy(X[idx]).unsqueeze(1)

preds = []
with torch.no_grad():
    for i in range(0, len(Xt), 256):
        xb = Xt[i:i+256].float().div_(255.).sub_(norm['mean']).div_(norm['std'])
        preds.append(F.softmax(model(xb), 1).argmax(1))
preds = torch.cat(preds).numpy()

wrong, wrong_as = idx[preds != TARGET], preds[preds != TARGET]
print(f'{CLASSES[TARGET]}: {len(wrong)}/{len(idx)} misclassified')

fig, axes = plt.subplots(2, 5, figsize=(13, 6))
rng = np.random.default_rng(0)
for ax, k in zip(axes.ravel(), rng.choice(len(wrong), size=min(10, len(wrong)), replace=False)):
    ax.imshow(X[wrong[k]], cmap='gray', vmin=0, vmax=255); ax.axis('off')
    ax.set_title(f'called {CLASSES[wrong_as[k]]}', fontsize=8)
plt.tight_layout()
plt.savefig('artifacts/figures/day2_mousebite_errors.png', dpi=130)
plt.show()

In [ ]:
snapshot()
